<a href="https://colab.research.google.com/github/aarongarza251/TC1004BM1A4V2/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
pip install pandas

In [5]:
pip install cryptography

In [6]:
import pandas as pd
print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.2


In [7]:
!pip install pymysql sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.1 MB/s eta 0:00:00


In [8]:
from google.colab import userdata

In [9]:
! pip install pymysql sqlalchemy
import pymysql
import os
import tempfile

# Define the SecretNotFoundError to catch it
class SecretNotFoundError(Exception):
    pass

timeout = 10

# Retrieve secrets with error handling for optional ones
host_secret = userdata.get('MYSQL_HOST')
user_secret = userdata.get('MYSQL_USER')
password_secret = userdata.get('MYSQL_PASSWORD')
database_secret = userdata.get('MYSQL_DATABASE')

port_secret = None
try:
    port_secret = userdata.get('MYSQL_PORT')
except Exception as e:
    if 'SecretNotFoundError' in str(type(e)):
        print("MYSQL_PORT secret not found. Defaulting to 3306.")
    else:
        raise e # Re-raise if it's an unexpected error

ssl_cert_content_secret = None
try:
    ssl_cert_content_secret = userdata.get('sslCert')
except Exception as e:
    if 'SecretNotFoundError' in str(type(e)):
        print("sslCert secret not found. Proceeding without SSL CA certificate in SSL context.")
    else:
        raise e # Re-raise if it's an unexpected error

# Validate essential secrets are not None or empty strings
if not all([host_secret, user_secret, password_secret, database_secret]):
    raise ValueError("One or more essential MySQL connection secrets (MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DATABASE) are missing or empty. Please check your Colab secrets.")

MYSQL_HOST = host_secret.strip()
MYSQL_USERNAME = user_secret.strip()
MYSQL_PASSWORD = password_secret
MYSQL_DATABASE = database_secret.strip()
MYSQL_PORT = int(port_secret) if port_secret is not None and str(port_secret).strip() != '' else 3306 # Default to 3306 if port secret is not set or empty

ca_cert_file_path = None
temp_dir = None
ssl_args = None

if ssl_cert_content_secret:
    try:
        temp_dir = tempfile.mkdtemp()
        ca_cert_file_path = os.path.join(temp_dir, 'ca.pem')
        with open(ca_cert_file_path, 'w') as f:
            f.write(ssl_cert_content_secret)
        print(f"SSL CA certificate written to temporary file: {ca_cert_file_path}")
        ssl_args = {'ca': ca_cert_file_path}
    except Exception as e:
        print(f"Error creating temporary SSL certificate file: {e}")
        print("Proceeding without SSL CA certificate in SSL context.")
else:
    print("No 'sslCert' secret found. Proceeding without SSL CA certificate in SSL context.")

connection = None
try:
    connection = pymysql.connect(
        charset="utf8mb4",
        connect_timeout=timeout,
        cursorclass=pymysql.cursors.DictCursor,
        db=MYSQL_DATABASE,
        host=MYSQL_HOST,
        password=MYSQL_PASSWORD,
        read_timeout=timeout,
        port=MYSQL_PORT,
        user=MYSQL_USERNAME,
        write_timeout=timeout,
        ssl=ssl_args # Pass SSL arguments here
    )
    print("Successfully connected to MySQL database!")

    # Example: Create a cursor object and execute a query
    with connection.cursor() as cursor:
        cursor.execute("SELECT VERSION();")
        version = cursor.fetchone()
        print(f"MySQL Server Version: {version['VERSION()']}")

except pymysql.Error as e:
    print(f"Error connecting to MySQL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Close the connection if it was successfully opened
    if connection and connection.open:
        print("Connection open succesfully")

/bin/bash: line 1:  pip: command not found
sslCert secret not found. Proceeding without SSL CA certificate in SSL context.
No 'sslCert' secret found. Proceeding without SSL CA certificate in SSL context.
Successfully connected to MySQL database!
MySQL Server Version: 8.0.45
Connection open succesfully


In [10]:
import pandas as pd

# Assuming 'connection' is an already established pymysql connection object
# and 'pymysql.cursors.DictCursor' is used for dictionary-like results.

if 'connection' not in locals() or not connection.open:
    print("Error: MySQL connection is not open. Please ensure the connection cell was run successfully.")
else:
    try:
        with connection.cursor() as cursor:
            print("\nAttempting to query table 'clientes' for a random 10-row sample...")
            cursor.execute("SELECT * FROM clientes ORDER BY RAND() LIMIT 10;")
            sample_data = cursor.fetchall()
            if sample_data:
                df = pd.DataFrame(sample_data)
                print("Random 10-row sample data from 'clientes' table:")
                display(df)
            else:
                print("Table 'clientes' is empty or no rows returned.")
    except pymysql.Error as e:
        print(f"Error querying 'clientes' table: {e}. It might not exist.")
    except Exception as e:
        print(f"An unexpected error occurred while querying 'clientes': {e}")


Attempting to query table 'clientes' for a random 10-row sample...
Random 10-row sample data from 'clientes' table:


,ID,Nombre,Apellido,Dirección,Teléfono,Correo Electrónico,Fecha de Registro,¿Cómo se enteró?,Código Postal,CURP
0,1030,Elena,Marroquín,"Circuito Grecia 876 Interior 484, San Iván los...",675.605.4272,elena@gmail.com,26/10/25,Facebook,26424,MAEL07MMDD00
1,1024,Camilo,Escamilla,"Boulevard Trinidad y Tabago 574 Interior 870, ...",05670020457,camilo@example.com,invalid_date,TikTok,45233-5850,ESCA70MMDD00
2,1023,Graciela,Olivárez,"Calle Querétaro 187 845, Nueva Austria, SLP 04...",04926770001,graciela@gmail.com,02/10/24,Instgram,93466,OLGR18MMDD00
3,1003,Nelly,Vargas,"Continuación Ceja 360 060, Nueva Malasia, AGS ...",140.550.5195,nelly@hotmail.com,invalid_date,Facebook,75595-7243,VANE73MMDD00
4,1008,Socorro,Saldaña,"Ampliación Omán 008 Interior 374, Nueva Aleman...",440-258-0699,socorro@gmail.com,"December 18, 2023",Recomendación,32572-1999,SASO10MMDD00
5,1029,Tomás,Aguayo,"Calle Sur Castro 221 Interior 858, San Adalber...",506-896-4210,tomás@example.com,04.08.2024,Instgram,56501-6626,AGTO09MMDD00
6,1014,Blanca,Farías,"Pasaje Guatemala 966 Edif. 825 , Depto. 093, V...",1-547-808-4558,blanca@hotmail.com,02/04/25,TikTok,43922-5215,FABL01MMDD00
7,1032,Daniel,Espino,"Viaducto Michoacán de Ocampo 120 Edif. 264 , D...",1-120-215-8525,daniel@gmail.com,10/05/25,TikTok,04828,ESDA71MMDD###
8,1001,Elvia,Guerra,"Calzada Gabón 585 Edif. 530 , Depto. 808, San ...",1-626-849-8301,elvia@yahoo.com,17/02/25,tiktok,17931,GUEL71MMDD###
9,1020,Eloy,Ríos,"Avenida Sur Tapia 337 Interior 388, Vieja Iraq...",781.886.8503,eloy@hotmail,invalid_date,gogle,69260-0868,RÍEL72MMDD###
